# 🧠 Classifying Waste Products with Transfer Learning and Fine-Tuning

In this project I build a binary image classifier that separates **organic** waste from **recyclable**
waste, using VGG16 pre-trained on ImageNet as the feature extractor.

I train it twice and compare the results: first with the entire base **frozen** (feature extraction),
then with one convolutional block **unfrozen** (fine-tuning). The comparison is the point — it shows
concretely what fine-tuning buys and what it costs.

The application is real: automated waste sorting for municipal or industrial recycling lines.

## 📋 Overview

Sorting waste is a classification problem with a hard operational constraint — it has to run fast and
cheap on a conveyor. That makes it a good fit for transfer learning: I don't have millions of labelled
waste images, and I don't need to train a vision model from scratch to get one.

| What I build | Why it exists |
|---|---|
| 📥 Waste Classification dataset | Real photos, folder-per-class (O = organic, R = recyclable) |
| 🔄 Data generators with augmentation | Shift and flip so the model tolerates real-world framing |
| 🏗️ VGG16 base, frozen | Borrowed ImageNet feature extractor |
| ➕ Dense head, sigmoid output | Binary decision layer — the only task-specific part |
| 🎯 Model A — feature extraction | Base fully frozen, only the head learns |
| 🔬 Model B — fine-tuned | `block5_conv3` onward unfrozen, adapts to waste imagery |
| 📊 Side-by-side evaluation | Classification report for both, on the same test images |
| 🧪 Individual predictions | Look at actual images, not just aggregate numbers |

**The ten graded tasks**, and where each lands in this notebook:

| Task | What it asks | Part |
|---|---|---|
| 1 | Print the TensorFlow version | Part 1 |
| 2 | Create the `test_generator` | Part 4 |
| 3 | Print `len(train_generator)` | Part 4 |
| 4 | Print the model summary | Part 5 |
| 5 | Compile the model | Part 5 |
| 6 | Accuracy curves, feature-extraction model | Part 7 |
| 7 | Loss curves, fine-tuned model | Part 9 |
| 8 | Accuracy curves, fine-tuned model | Part 9 |
| 9 | Plot a test image, feature-extraction model | Part 11 |
| 10 | Plot a test image, fine-tuned model | Part 11 |

## 🧩 Theory

### 🏗️ What transfer learning reuses

A convolutional network learns a hierarchy of visual features. The early layers detect edges, colour
gradients and textures — those are **generic**, useful for any image task. Only the late layers encode
something specific to the original training categories.

So the recipe is: keep the early layers, discard the original classifier, and train a new head:

```
VGG16 (ImageNet)              This project
──────────────────            ─────────────────────────────
block1_conv  edges       ❄️ frozen  ─┐
block2_conv  textures    ❄️ frozen   │  generic — reused as-is
block3_conv  motifs      ❄️ frozen   │
block4_conv  parts       ❄️ frozen  ─┘
block5_conv  objects     ❄️ / 🔥      ← frozen in model A, unfrozen in model B
──────────────────
1000-class head          ✂️ discarded (include_top=False)
                         ➕ replaced with: Dense→Dropout→Dense→Dropout→Dense(1, sigmoid)
```

I don't design an RF front-end from scratch for every product — the amplifier, mixer and filter chain
is generic and gets reused, and only the final stage is retuned for the specific band. Same structure.

### ❄️ vs 🔬 Feature extraction and fine-tuning

These are the two modes of transfer learning, and this notebook does both so they can be compared:

| | **Model A — feature extraction** | **Model B — fine-tuning** |
|---|---|---|
| Base layers | All frozen ❄️ | `block5_conv3` onward trainable 🔥 |
| Trainable params | Head only | Head + last conv block |
| Optimizer | Adam | RMSprop |
| Risk | Underfits if the domain differs from ImageNet | Overfits, or damages pre-trained weights |
| Best when | Small dataset, similar domain | More data, or a domain ImageNet doesn't cover well |

Waste photos aren't in ImageNet's 1000 categories, but they're ordinary objects photographed
ordinarily — so ImageNet features transfer reasonably well. That's why feature extraction alone gets a
sensible result, and fine-tuning adds a modest improvement rather than a dramatic one.

### 🔢 Binary classification: sigmoid and the decision threshold

Two classes, so the output layer is a **single neuron with sigmoid**, not two neurons with softmax:

$$\sigma(z) = \frac{1}{1 + e^{-z}}$$

Sigmoid squashes any real number into $(0, 1)$, read as $P(\text{recyclable})$. The hard decision comes
from thresholding:

$$\hat{y} = \begin{cases}
\text{R (recyclable)} & \text{if } \sigma(z) \geq 0.5 \\[4pt]
\text{O (organic)} & \text{if } \sigma(z) < 0.5
\end{cases}$$

This is exactly a **soft-decision demodulator followed by a hard-decision slicer**. The sigmoid output
is the soft value carrying confidence; the 0.5 comparison is the slicer that commits to a symbol. And
as in a receiver, the threshold is a *choice* — moving it trades false positives against false
negatives. On a real sorting line, sending a recyclable to landfill and sending organic waste to the
recycling stream have different costs, so 0.5 wouldn't necessarily be the operating point.

### 📉 Binary cross-entropy

$$\mathcal{L} = -\big[\,y \log(\hat{y}) + (1 - y)\log(1 - \hat{y})\,\big]$$

Where $y \in \{0, 1\}$ is the true label and $\hat{y}$ the sigmoid output. Only one term survives per
sample: if $y=1$ the loss is $-\log(\hat{y})$, if $y=0$ it's $-\log(1-\hat{y})$. Either way it's the
negative log of the probability assigned to the truth, so confident mistakes are punished steeply.

| Task type | Output layer | Loss |
|---|---|---|
| **Binary** (this project) | `Dense(1, sigmoid)` | `binary_crossentropy` |
| Multi-class, one-hot | `Dense(C, softmax)` | `categorical_crossentropy` |
| Multi-class, integer labels | `Dense(C, softmax)` | `sparse_categorical_crossentropy` |

### 📉 Exponential learning rate decay

The scheduler reduces the learning rate every epoch on a fixed exponential curve:

$$\eta(t) = \eta_0 \cdot e^{-kt}, \qquad \eta_0 = 10^{-4},\; k = 0.1$$

| Epoch $t$ | $\eta(t)$ |
|---|---|
| 0 | $1.00 \times 10^{-4}$ |
| 2 | $8.19 \times 10^{-5}$ |
| 5 | $6.07 \times 10^{-5}$ |
| 9 | $4.07 \times 10^{-5}$ |

Same exponential shape as a forgetting factor in an EWMA — big steps early to cover ground, small steps
later to settle without overshooting.

⚠️ **This schedule silently overrides whatever learning rate is passed to the optimizer.** More on that
in Part 6; it's the most consequential problem in the source code.

### ⚙️ The callbacks

| Callback | Watches | Behaviour |
|---|---|---|
| `LearningRateScheduler(exp_decay)` | Epoch number | Sets LR from the formula above, every epoch |
| `EarlyStopping(patience=4, min_delta=0.01)` | `val_loss` | Stops after 4 epochs without a 0.01 improvement |
| `ModelCheckpoint(save_best_only=True)` | `val_loss` | Writes the best model to disk |
| `LossHistory_` (custom) | — | Records loss and LR each epoch, prints the LR |

`ModelCheckpoint` matters more than it looks. Training ends holding the *last* epoch's weights, which
aren't necessarily the best. Because the checkpoint file holds the best-scoring version, Part 10
reloads from disk rather than using the in-memory model.

## Part 1 — ⚙️ Setup and Imports

Pinned library versions first. TensorFlow, NumPy and scikit-learn have genuine cross-version
incompatibilities, so fixing a known-good combination avoids confusing failures later.

In [ ]:
!pip install tensorflow==2.17.0 | tail -n 1
!pip install numpy | tail -n 1
!pip install scikit-learn==1.5.1  | tail -n 1
!pip install matplotlib==3.9.2  | tail -n 1

In [ ]:
import numpy as np
import os
# import random, shutil
import glob


from matplotlib import pyplot as plt
from matplotlib import pyplot
from matplotlib.image import imread

# from os import makedirs,listdir
# from shutil import copyfile
# from random import seed
# from random import random

os.environ['TF_CPP_MIN_LOG_LEVEL'] = '3'

import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras import optimizers
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint
# from tensorflow.keras.layers import Conv2D, MaxPooling2D,GlobalAveragePooling2D, Input
from tensorflow.keras.layers import Dense, Dropout, Flatten
from tensorflow.keras.models import Sequential, Model
from tensorflow.keras.preprocessing.image import ImageDataGenerator
# from tensorflow.keras.applications import InceptionV3
from sklearn import metrics

import warnings
warnings.filterwarnings('ignore')

📝 **What matters in that import block:**

| Import | Role |
|---|---|
| `ImageDataGenerator` | Loads images from folders in batches, applies augmentation |
| `Sequential` / `Model` | Two model-building APIs — I use both, for different reasons (Part 5) |
| `Dense`, `Dropout`, `Flatten` | The classification head |
| `EarlyStopping`, `ModelCheckpoint` | Stop when it stops improving; keep the best weights |
| `sklearn.metrics` | `classification_report` for precision/recall/F1 in Part 10 |
| `TF_CPP_MIN_LOG_LEVEL = '3'` | Silences TensorFlow's C++ logging — cosmetic only |

The commented-out imports are left exactly as the source had them.

### ✅ Task 1 — Print the TensorFlow version

Version matters more than it might seem. Keras 3 shipped with TensorFlow 2.16 and changed several
behaviours, so knowing which version is running is the first thing to check when something behaves
unexpectedly.

In [ ]:
# Task 1: Print the version of tensorflow
print(tf.__version__)

## Part 2 — 📥 Getting the Waste Dataset

The dataset is a reduced version of the Kaggle Waste Classification set — photographs sorted into two
folders, `O` (organic) and `R` (recyclable). The label is the folder name; there is no separate
annotation file.

Download uses `requests` with `stream=True` so the archive is written in 8 KB chunks rather than held
in memory all at once, and `tqdm` gives a progress bar during extraction.

In [ ]:
import requests
import zipfile
from tqdm import tqdm

url = "https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/kd6057VPpABQ2FqCbgu9YQ/o-vs-r-split-reduced-1200.zip"
file_name = "o-vs-r-split-reduced-1200.zip"

print("Downloading file")
with requests.get(url, stream=True) as response:
    response.raise_for_status()
    with open(file_name, 'wb') as f:
        for chunk in response.iter_content(chunk_size=8192):
            f.write(chunk)


def extract_file_with_progress(file_name):
    print("Extracting file with progress")
    with zipfile.ZipFile(file_name, 'r') as zip_ref:
        members = zip_ref.infolist()
        with tqdm(total=len(members), unit='file') as progress_bar:
            for member in members:
                zip_ref.extract(member)
                progress_bar.update(1)
    print("Finished extracting file")


extract_file_with_progress(file_name)

print("Finished extracting file")
os.remove(file_name)

📝 **Notes:**

- `response.raise_for_status()` turns an HTTP error into a Python exception instead of silently writing
  an error page to disk as if it were a zip.
- `stream=True` plus chunked writing keeps memory flat regardless of archive size.
- `os.remove(file_name)` deletes the zip after extraction to save space.

⚠️ **Not idempotent.** Unlike a guarded download, this cell re-downloads and re-extracts every time it
runs. Re-running it wastes bandwidth and time. Adding `if not os.path.exists('o-vs-r-split'):` around
it would fix that — Sandbox exercise 6.

The resulting layout:

```
o-vs-r-split/
├── train/
│   ├── O/     ← organic
│   └── R/     ← recyclable
└── test/
    ├── O/
    └── R/
```

## Part 3 — 🔢 Configuration

All the knobs in one place, which makes them easy to sweep later.

In [ ]:
img_rows, img_cols = 150, 150
batch_size = 32
n_epochs = 10
n_classes = 2
val_split = 0.2
verbosity = 1
path = 'o-vs-r-split/train/'
path_test = 'o-vs-r-split/test/'
input_shape = (img_rows, img_cols, 3)
labels = ['O', 'R']
seed = 42

📝 **What each setting does:**

| Parameter | Value | Effect |
|---|---|---|
| `img_rows, img_cols` | 150 × 150 | Every image resized to this |
| `batch_size` | 32 | Images per gradient step |
| `n_epochs` | 10 | Upper bound — `EarlyStopping` may cut it short |
| `val_split` | 0.2 | 20% of training data held out for validation |
| `labels` | `['O', 'R']` | Alphabetical, so O → 0 and R → 1 |
| `seed` | 42 | Makes the train/validation split reproducible |

🔍 **Why 150 × 150 gives a 4 × 4 feature map.** VGG16 has five max-pooling stages, each halving both
spatial dimensions, with integer floor division at each step:

$$150 \to 75 \to 37 \to 18 \to 9 \to 4$$

So the final convolutional output is **4 × 4 × 512**, and flattening gives:

$$4 \times 4 \times 512 = 8192 \text{ features}$$

That 8192 is what feeds the first `Dense(512)`, which is where most of this model's parameters live —
about 4.2 million in that one layer alone.

⚠️ `n_classes = 2` and `n_epochs` are defined here but never referenced; the model hardcodes a single
sigmoid output and `epochs=10` is passed literally to `fit()`. Harmless, but they're decoration rather
than configuration.

## Part 4 — 🔄 Data Generators

Three generators. Only the training one augments — validation and test get rescaling alone, because
measuring performance on deliberately distorted images would tell me nothing useful.

**Rescaling** maps pixels from $[0, 255]$ to $[0, 1]$:

$$x_{\text{scaled}} = \frac{x}{255}$$

Large-magnitude inputs saturate activations and make gradients behave badly. Same instinct as keeping
a signal inside an ADC's linear region rather than driving it into clipping.

In [ ]:
# Create ImageDataGenerators for training and validation and testing
train_datagen = ImageDataGenerator(
    validation_split = val_split,
    rescale=1.0/255.0,
	width_shift_range=0.1,
    height_shift_range=0.1,
    horizontal_flip=True
)

val_datagen = ImageDataGenerator(
    validation_split = val_split,
    rescale=1.0/255.0,
)

test_datagen = ImageDataGenerator(
    rescale=1.0/255.0
)

📝 **The augmentation is deliberately mild** — ±10% shift and a horizontal flip, nothing more. No
rotation, no zoom, no shear.

That restraint is appropriate here. Waste photos are taken roughly upright and roughly framed; heavy
rotation would produce images the model will never encounter in deployment. Augmentation should
simulate the variation that actually occurs, not variation for its own sake. This is the same
discipline as choosing which channel impairments to test a receiver against — you model the ones the
link will really see.

⚠️ `validation_split` is set on **both** `train_datagen` and `val_datagen`, and both must match, since
the split is computed per-generator from the same directory. They do (`val_split = 0.2`), so the
`subset='training'` and `subset='validation'` calls below carve out disjoint sets.

In [ ]:
train_generator = train_datagen.flow_from_directory(
    directory = path,
    seed = seed,
    batch_size = batch_size,
    class_mode='binary',
    shuffle = True,
    target_size=(img_rows, img_cols),
    subset = 'training'
)

In [ ]:
val_generator = val_datagen.flow_from_directory(
    directory = path,
    seed = seed,
    batch_size = batch_size,
    class_mode='binary',
    shuffle = True,
    target_size=(img_rows, img_cols),
    subset = 'validation'
)

📝 `class_mode='binary'` produces a single 0/1 label per image rather than a one-hot vector — which is
what pairs with a sigmoid output and `binary_crossentropy`.

### ✅ Task 2 — Create the `test_generator`

The test generator differs from the other two in one important way: **`shuffle=False`**.

That's not a style preference. Part 10 compares predictions against labels by position, so the order
images come out in has to be stable and match the label list. Shuffling the test set would silently
scramble that alignment — predictions and labels would still have the same *length*, so nothing would
error, and the metrics would just be wrong. Another failure that looks like success.

In [ ]:
# Task 2: Create a `test_generator` using the `test_datagen` object
test_generator = test_datagen.flow_from_directory(
    directory=path_test,
    class_mode='binary',
    seed=seed,
    batch_size=batch_size,
    shuffle=False,
    target_size=(img_rows, img_cols)
)

### ✅ Task 3 — Print the length of the `train_generator`

`len(generator)` returns the number of **batches per epoch**, not the number of images:

$$\text{len(generator)} = \left\lceil \frac{\text{number of images}}{\text{batch size}} \right\rceil$$

This is the number worth knowing before setting `steps_per_epoch` — it's the value that would make one
epoch equal one full pass over the training data.

In [ ]:
# Task 3: print the length of the `train_generator`
print(len(train_generator))

Before building the model, a quick look at what augmentation actually produces. Five variants of a
single organic-waste image, each with a different random shift and flip:

In [ ]:
from pathlib import Path

IMG_DIM = (100, 100)

train_files = glob.glob('./o-vs-r-split/train/O/*')
train_files = train_files[:20]
train_imgs = [tf.keras.preprocessing.image.img_to_array(tf.keras.preprocessing.image.load_img(img, target_size=IMG_DIM)) for img in train_files]
train_imgs = np.array(train_imgs)
train_labels = [Path(fn).parent.name for fn in train_files]

img_id = 0
O_generator = train_datagen.flow(train_imgs[img_id:img_id+1], train_labels[img_id:img_id+1],
                                   batch_size=1)
O = [next(O_generator) for i in range(0,5)]
fig, ax = plt.subplots(1,5, figsize=(16, 6))
print('Labels:', [item[1][0] for item in O])
l = [ax[i].imshow(O[i][0][0]) for i in range(0,5)]

📝 The five images are the *same source photo* transformed five different ways. The model sees a
different version every epoch, which is what stops it memorising individual training pictures.

Note this preview uses `IMG_DIM = (100, 100)`, unrelated to the 150 × 150 the model actually trains
on — it's for display only.

## Part 5 — 🏗️ Building the Feature-Extraction Model

Load VGG16 without its classifier head, at the 150 × 150 input size configured earlier.

In [ ]:
from tensorflow.keras.applications import vgg16

input_shape = (150, 150, 3)
vgg = vgg16.VGG16(include_top=False,
                        weights='imagenet',
                        input_shape=input_shape)

Now take VGG16's final output, flatten it, and wrap the whole thing as a reusable `Model` object:

- **inputs:** `vgg.input`
- **outputs:** `tf.keras.layers.Flatten()(output)`

This is the **Functional API** — needed here because I'm grafting a new layer onto an existing model's
output, which `Sequential` can't express. The head that comes afterward *is* built with `Sequential`,
since it's a plain linear stack. Using both APIs in one model is normal.

In [ ]:
output = vgg.layers[-1].output
output = tf.keras.layers.Flatten()(output)
basemodel = Model(vgg.input, output)

Freeze every layer in the base. The head starts with random weights, so its first gradients are large
and meaningless — letting those flow back into ImageNet-tuned weights would damage exactly the thing
worth borrowing.

In [ ]:
for layer in basemodel.layers:
    layer.trainable = False

Now the classification head on top: two `Dense(512)` layers with `Dropout(0.3)` between them, ending
in a single sigmoid unit.

In [ ]:
input_shape = basemodel.output_shape[1]

model = Sequential()
model.add(basemodel)
model.add(Dense(512, activation='relu'))
model.add(Dropout(0.3))
model.add(Dense(512, activation='relu'))
model.add(Dropout(0.3))
model.add(Dense(1, activation='sigmoid'))

📝 **The head, layer by layer:**

| Layer | Output | Why |
|---|---|---|
| `basemodel` | 8192 | Frozen VGG16 features, flattened |
| `Dense(512, relu)` | 512 | ~4.2M parameters — where the capacity is |
| `Dropout(0.3)` | 512 | Zeroes 30% of activations per step |
| `Dense(512, relu)` | 512 | Second layer, ~262k parameters |
| `Dropout(0.3)` | 512 | More regularisation |
| `Dense(1, sigmoid)` | 1 | $P(\text{recyclable})$ |

⚠️ `input_shape = basemodel.output_shape[1]` is assigned and then never used — `Sequential` infers the
shape from the layer it's given. It also shadows the `input_shape` tuple set earlier, which is
confusing to read even though nothing downstream depends on it.

🔍 **`Flatten` here, not `GlobalAveragePooling2D`.** Flattening 4 × 4 × 512 gives 8192 features and a
4.2-million-parameter first Dense layer. Global average pooling would give 512 features and about
262k — sixteen times fewer. Flatten preserves *where* in the frame each feature fired, which pooling
discards. For distinguishing an apple core from a plastic bottle, position almost certainly doesn't
matter, so this is a lot of parameters bought for little. Sandbox exercise 3 tests that.

### ✅ Task 4 — Print the model summary

`model.summary()` is the fastest way to confirm the architecture is what I intended. Check three
things: the output shapes make sense, the total parameter count is plausible, and — most importantly
here — **trainable vs non-trainable** matches what freezing was supposed to achieve. With the base
frozen, roughly 14.7M parameters should show as non-trainable.

In [ ]:
# Task 4: print the summary of the model
model.summary()

### ✅ Task 5 — Compile the model

Three settings, and they have to be mutually consistent:

| Setting | Value | Why |
|---|---|---|
| `loss` | `binary_crossentropy` | Two classes, single sigmoid output |
| `optimizer` | `Adam(learning_rate=1e-5)` | Small steps — the head sits on frozen features |
| `metrics` | `['accuracy']` | Reported, not optimised |

The `for layer ... trainable = False` loop is repeated here from the source. It's redundant — the base
was already frozen — but harmless, and re-asserting it immediately before `compile()` is defensible
since compile is what actually locks the trainable set in.

In [ ]:
for layer in basemodel.layers:
    layer.trainable = False

# Task 5: Compile the model
model.compile(
    loss='binary_crossentropy',
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-5),
    metrics=['accuracy']
)

## Part 6 — 🎯 Training the Feature-Extraction Model

The callbacks are defined here: a custom history recorder, the exponential LR schedule, early stopping
and best-model checkpointing.

In [ ]:
from tensorflow.keras.callbacks import LearningRateScheduler


checkpoint_path='O_R_tlearn_vgg16.keras'

# define step decay function
class LossHistory_(tf.keras.callbacks.Callback):
    def on_train_begin(self, logs={}):
        self.losses = []
        self.lr = []

    def on_epoch_end(self, epoch, logs={}):
        self.losses.append(logs.get('loss'))
        self.lr.append(exp_decay(epoch))
        print('lr:', exp_decay(len(self.losses)))

def exp_decay(epoch):
    initial_lrate = 1e-4
    k = 0.1
    lrate = initial_lrate * np.exp(-k*epoch)
    return lrate

# learning schedule callback
loss_history_ = LossHistory_()
lrate_ = LearningRateScheduler(exp_decay)

keras_callbacks = [
      EarlyStopping(monitor = 'val_loss',
                    patience = 4,
                    mode = 'min',
                    min_delta=0.01),
      ModelCheckpoint(checkpoint_path, monitor='val_loss', save_best_only=True, mode='min')
]

callbacks_list_ = [loss_history_, lrate_] + keras_callbacks

⚠️ **Bug 1 — the learning rate scheduler silently overrides the optimizer.**

Task 5 compiled with `Adam(learning_rate=1e-5)`. But `LearningRateScheduler(exp_decay)` sets the
learning rate at the start of **every epoch**, from `exp_decay(epoch)`:

$$\eta(0) = 10^{-4} \cdot e^{0} = 10^{-4}$$

So from epoch 0 the actual learning rate is $10^{-4}$ — **ten times larger** than what was compiled in.
The `1e-5` never takes effect for a single step.

This is the same class of bug as two decay mechanisms fighting over one variable: whoever writes last
wins, and the value in the code you read isn't the value the model uses. The scheduler is not wrong to
do this — that's its job — but having both is a contradiction, and the `1e-5` is misleading to anyone
reading the compile call.

Either drop the scheduler and let `Adam(1e-5)` stand, or set `initial_lrate = 1e-5` in `exp_decay` so
the two agree. Sandbox exercise 1.

⚠️ **Bug 3 — the `LossHistory_` callback is off by one.** `on_epoch_end` appends `exp_decay(epoch)` to
its list but *prints* `exp_decay(len(self.losses))`, which is `epoch + 1` after the append. So the
printed LR is next epoch's value, not the one just used. Cosmetic, but the number on screen isn't the
number that trained the model.

| Callback | Setting | Behaviour |
|---|---|---|
| `EarlyStopping` | `patience=4`, `min_delta=0.01` | Needs a 0.01 improvement in `val_loss` to count as progress |
| `ModelCheckpoint` | `save_best_only=True` | Writes `O_R_tlearn_vgg16.keras` whenever `val_loss` improves |

Note `EarlyStopping` here has **no** `restore_best_weights=True`. That's fine only because
`ModelCheckpoint` is saving the best model separately and Part 10 loads from that file. Without the
checkpoint, the in-memory model at the end of training could be worse than the best one seen.

In [ ]:
extract_feat_model = model.fit(train_generator,
                               steps_per_epoch=5,
                               epochs=10,
                               callbacks = callbacks_list_,
                               validation_data=val_generator,
                               validation_steps=val_generator.samples // batch_size,
                               verbose=1)

⚠️ **Bug 2 — the training budget is tiny.**

$$\text{images per epoch} = \text{steps\_per\_epoch} \times \text{batch\_size} = 5 \times 32 = 160$$

160 images per epoch, 10 epochs — 1600 image-presentations total, and `len(train_generator)` from
Task 3 shows how small a fraction of one epoch that is. This is a demonstration budget, not a training
budget.

It interacts badly with `EarlyStopping(patience=4, min_delta=0.01)`: with so few steps, `val_loss`
moves erratically between epochs, so the callback may well fire on noise rather than on genuine
convergence. Expect modest and variable accuracy.

Removing `steps_per_epoch` entirely would make each epoch a full pass over the data — Sandbox exercise 2.

## Part 7 — 📈 Curves for the Feature-Extraction Model

Loss first, then accuracy. Curves show things a final number can't — in particular the *gap* between
training and validation, which is the clearest overfitting signal available.

In [ ]:
import matplotlib.pyplot as plt

history = extract_feat_model

# plot loss curve
plt.figure(figsize=(5, 5))
plt.plot(history.history['loss'], label='Training Loss')
plt.plot(history.history['val_loss'], label='Validation Loss')
plt.title('Loss Curve')
plt.xlabel('Epochs')
plt.ylabel('Loss')
plt.legend()

plt.show()

### ✅ Task 6 — Accuracy curves (feature-extraction model)

Same structure as the loss plot, reading `accuracy` and `val_accuracy` from the same history object.

Loss and accuracy are **not** redundant. Loss is what gradient descent actually minimises and is
sensitive to confidence; accuracy only counts whether the thresholded decision was right. A model can
improve loss while accuracy stays flat — it's getting more confident about things it already had
correct. Watching both distinguishes real progress from mere confidence.

In [ ]:
import matplotlib.pyplot as plt

history = extract_feat_model
## Task 6: Plot accuracy curves for training and validation sets
plt.figure(figsize=(5, 5))
plt.plot(history.history['accuracy'], label='Training Accuracy')
plt.plot(history.history['val_accuracy'], label='Validation Accuracy')
plt.title('Accuracy Curve')
plt.xlabel('Epochs')
plt.ylabel('Accuracy')
plt.legend()

plt.show()

📝 **Reading the curves:**

| Pattern | Diagnosis | Response |
|---|---|---|
| Both rising together | 🟢 Learning | Train longer |
| Training ↑, validation flat or ↓ | 🔴 [[overfitting]] | More dropout/augmentation/data |
| Both flat and low | 🟡 [[underfitting]] | Bigger head, higher LR, unfreeze layers |
| Very noisy validation | 🟡 Too few validation steps | Larger validation set |

With only 160 images per epoch, expect the noisy case. Judge the trend, not individual points.

## Part 8 — 🔬 Fine-Tuning: Unfreezing `block5_conv3`

Now model B. Everything is rebuilt from scratch — a fresh VGG16, a fresh head — so the two models are
independent and comparable rather than one continuing from the other.

The difference is the freezing logic. Instead of freezing everything, it walks the layer list and flips
a `set_trainable` flag when it reaches `block5_conv3`. From that layer onward everything is trainable;
everything before stays frozen.

This targets the *last* convolutional layer of the *last* block — the most abstract, most
ImageNet-specific features, and therefore the ones that most benefit from adapting to waste imagery.
The generic edge and texture detectors underneath are already correct and are left alone.

In [ ]:
from tensorflow.keras.applications import vgg16

input_shape = (150, 150, 3)
vgg = vgg16.VGG16(include_top=False,
                        weights='imagenet',
                        input_shape=input_shape)

output = vgg.layers[-1].output
output = tf.keras.layers.Flatten()(output)
basemodel = Model(vgg.input, output)

for layer in basemodel.layers:
    layer.trainable = False

display([layer.name for layer in basemodel.layers])

set_trainable = False

for layer in basemodel.layers:
    if layer.name in ['block5_conv3']:
        set_trainable = True
    if set_trainable:
        layer.trainable = True
    else:
        layer.trainable = False

for layer in basemodel.layers:
    print(f"{layer.name}: {layer.trainable}")

📝 **The flag pattern.** `set_trainable` starts `False` and flips to `True` the moment the loop hits
`block5_conv3`. Since Keras stores layers in forward order, everything from that point on becomes
trainable. It's a clean way to express "unfreeze from here down" without hardcoding an index — if the
architecture changed, a name still resolves correctly where `layers[-3:]` might not.

The final loop prints every layer with its trainable state, which is worth actually reading. Freezing
is exactly the kind of thing that fails silently, and this is the read-back that confirms the write
landed.

⚠️ **Bug 4 — `display()` is never imported.** It works because IPython injects `display` into the
notebook namespace automatically. Run this file as a plain Python script and it raises `NameError`.
`print()` would be portable; `from IPython.display import display` would be explicit. Minor, but it's
an invisible dependency on the execution environment.

Now the same head architecture as before, its own checkpoint file, and fresh callback instances.

In [ ]:
model = Sequential()
model.add(basemodel)
model.add(Dense(512, activation='relu'))
model.add(Dropout(0.3))
model.add(Dense(512, activation='relu'))
model.add(Dropout(0.3))
model.add(Dense(1, activation='sigmoid'))

checkpoint_path='O_R_tlearn_fine_tune_vgg16.keras'

# learning schedule callback
loss_history_ = LossHistory_()
lrate_ = LearningRateScheduler(exp_decay)

keras_callbacks = [
      EarlyStopping(monitor = 'val_loss',
                    patience = 4,
                    mode = 'min',
                    min_delta=0.01),
      ModelCheckpoint(checkpoint_path, monitor='val_loss', save_best_only=True, mode='min')
]

callbacks_list_ = [loss_history_, lrate_] + keras_callbacks

model.compile(loss='binary_crossentropy',
              optimizer=optimizers.RMSprop(learning_rate=1e-4),
              metrics=['accuracy'])

fine_tune_model = model.fit(train_generator,
                    steps_per_epoch=5,
                    epochs=10,
                    callbacks = callbacks_list_,
                    validation_data=val_generator,
                    validation_steps=val_generator.samples // batch_size,
                    verbose=1)

📝 **Two deliberate differences from model A:**

| | Model A | Model B |
|---|---|---|
| Optimizer | `Adam(1e-5)` | `RMSprop(1e-4)` |
| Trainable base layers | None | `block5_conv3` onward |
| Checkpoint file | `O_R_tlearn_vgg16.keras` | `O_R_tlearn_fine_tune_vgg16.keras` |

Separate checkpoint paths matter — reusing one would have model B silently overwrite model A, and
Part 10 would compare a model against itself.

⚠️ **The LR conflict applies here too, and the discrepancy is smaller but still real.** RMSprop is
constructed with `1e-4`, and `exp_decay(0)` also returns `1e-4` — so at epoch 0 they happen to agree.
From epoch 1 onward the scheduler takes over and decays it. The value isn't wrong here; the point is
that the optimizer's argument is decorative either way. Whatever number is written there, the
scheduler overwrites it.

Re-creating `loss_history_`, `lrate_` and the callback list is necessary rather than tidy-minded:
`LossHistory_` accumulates state in `self.losses`, so reusing the model-A instance would append model
B's history onto model A's.

## Part 9 — 📈 Curves for the Fine-Tuned Model

### ✅ Task 7 — Loss curves (fine-tuned model)

Identical structure to Part 7, but reading `fine_tune_model`. Comparing the two loss curves side by
side is the whole reason for training twice.

In [ ]:
history = fine_tune_model

## Task 7: Plot loss curves for training and validation sets (fine tune model)
plt.figure(figsize=(5, 5))
plt.plot(history.history['loss'], label='Training Loss')
plt.plot(history.history['val_loss'], label='Validation Loss')
plt.title('Loss Curve')
plt.xlabel('Epochs')
plt.ylabel('Loss')
plt.legend()

plt.show()

### ✅ Task 8 — Accuracy curves (fine-tuned model)

In [ ]:
history = fine_tune_model

# Task 8: Plot accuracy curves for training and validation sets  (fine tune model)
plt.figure(figsize=(5, 5))
plt.plot(history.history['accuracy'], label='Training Accuracy')
plt.plot(history.history['val_accuracy'], label='Validation Accuracy')
plt.title('Accuracy Curve')
plt.xlabel('Epochs')
plt.ylabel('Accuracy')
plt.legend()

plt.show()

📝 **What to compare between the two models:**

| Observation | What it suggests |
|---|---|
| Fine-tuned reaches lower validation loss | Adapting `block5` genuinely helped |
| Fine-tuned train/validation gap is wider | Extra trainable capacity is overfitting 160 images/epoch |
| Curves nearly identical | ImageNet features were already sufficient — fine-tuning added little |
| Fine-tuned is worse | LR too high for the unfrozen layers, or too little data |

Any of these is a legitimate outcome at this training budget. Training is stochastic and the sample is
small, so the honest read is the *trend*, and a single run isn't strong evidence either way.

## Part 10 — 📊 Evaluating Both Models on Test Data

Now the real comparison, on 100 held-out images the models have never seen — 50 organic, 50 recyclable.

The two saved checkpoints are **reloaded from disk** rather than used in memory. That's deliberate:
`save_best_only=True` means those files hold the best-scoring epoch, whereas the in-memory models hold
whatever the last epoch produced. Since `EarlyStopping` here has no `restore_best_weights`, reloading
is the only way to evaluate the best version.

In [ ]:
from pathlib import Path

# Load saved models
extract_feat_model = tf.keras.models.load_model('O_R_tlearn_vgg16.keras')
fine_tune_model = tf.keras.models.load_model('O_R_tlearn_fine_tune_vgg16.keras')

IMG_DIM = (150, 150)

# Load test images
test_files_O = glob.glob('./o-vs-r-split/test/O/*')
test_files_R = glob.glob('./o-vs-r-split/test/R/*')
test_files = test_files_O[:50] + test_files_R[:50]

test_imgs = [tf.keras.preprocessing.image.img_to_array(tf.keras.preprocessing.image.load_img(img, target_size=IMG_DIM)) for img in test_files]
test_imgs = np.array(test_imgs)
test_labels = [Path(fn).parent.name for fn in test_files]

# Standardize
test_imgs_scaled = test_imgs.astype('float32')
test_imgs_scaled /= 255

class2num_lt = lambda l: [0 if x == 'O' else 1 for x in l]
num2class_lt = lambda l: ['O' if x < 0.5 else 'R' for x in l]

test_labels_enc = class2num_lt(test_labels)

# Make predictions for both models
predictions_extract_feat_model = extract_feat_model.predict(test_imgs_scaled, verbose=0)
predictions_fine_tune_model = fine_tune_model.predict(test_imgs_scaled, verbose=0)

# Convert predictions to class labels
predictions_extract_feat_model = num2class_lt(predictions_extract_feat_model)
predictions_fine_tune_model = num2class_lt(predictions_fine_tune_model)

# Print classification report for both models
print('Extract Features Model')
print(metrics.classification_report(test_labels, predictions_extract_feat_model))
print('Fine-Tuned Model')
print(metrics.classification_report(test_labels, predictions_fine_tune_model))

📝 **The critical detail: preprocessing must mirror training exactly.**

`test_imgs_scaled /= 255` reproduces what `ImageDataGenerator(rescale=1.0/255.0)` did during training.
Skip it and the model receives $[0, 255]$ inputs when it learned on $[0, 1]$ — no error, no warning,
just confident nonsense. This is the single most common silent bug in an inference path.

**`num2class_lt` is the decision threshold** made explicit:

```python
num2class_lt = lambda l: ['O' if x < 0.5 else 'R' for x in l]
```

That `0.5` is the slicer from the Theory section. It's a choice, not a law — and on a real sorting line
it's the parameter you'd tune against the relative cost of each error type.

**Reading `classification_report`:**

| Metric | Meaning | When it matters most |
|---|---|---|
| **Precision** | Of everything called R, how much really was R | High cost of contaminating the recycling stream |
| **Recall** | Of all true R items, how many were caught | High cost of sending recyclables to landfill |
| **F1** | Harmonic mean of the two | Single balanced number |
| **Support** | Images per class | Confirms the 50/50 balance |

With a balanced test set, accuracy is meaningful. On an imbalanced one it wouldn't be — a model
predicting the majority class every time would score well and be useless.

⚠️ **Variable shadowing.** `extract_feat_model` and `fine_tune_model` were `History` objects from
`fit()`; this cell reassigns them to loaded `Model` objects. So the training curves in Parts 7 and 9
must be plotted *before* this cell runs. Re-running the notebook out of order breaks them with a
confusing error, since a `Model` has no `.history`.

## Part 11 — 🧪 Inspecting Individual Predictions

Aggregate metrics hide the interesting cases. A helper to display one image with its true and
predicted labels:

In [ ]:
# Plot one of the images with actual label and predicted label as title
def plot_image_with_title(image, model_name, actual_label, predicted_label):
    plt.imshow(image)
    plt.title(f"Model: {model_name}, Actual: {actual_label}, Predicted: {predicted_label}")
    plt.axis('off')
    plt.show()

# Specify index of image to plot, for example index 0
index_to_plot = 0
plot_image_with_title(
    image=test_imgs[index_to_plot].astype('uint8'),
    model_name='Extract Features Model',
    actual_label=test_labels[index_to_plot],
    predicted_label=predictions_extract_feat_model[index_to_plot],
    )

📝 Note it displays `test_imgs`, not `test_imgs_scaled` — the unscaled version cast back to `uint8`.
Matplotlib expects either `uint8` in $[0, 255]$ or float in $[0, 1]$; handing it the scaled float array
would work too, but the raw array cast to `uint8` is the clearer choice for display. The *model* still
gets the scaled version.

### ✅ Task 9 — Plot a test image using the Extract Features Model (`index_to_plot = 1`)

In [ ]:
# Task 9: Plot a test image using Extract Features Model (index_to_plot = 1)
index_to_plot = 1
plot_image_with_title(
    image=test_imgs[index_to_plot].astype('uint8'),
    model_name='Extract Features Model',
    actual_label=test_labels[index_to_plot],
    predicted_label=predictions_extract_feat_model[index_to_plot],
    )

### ✅ Task 10 — Plot the same test image using the Fine-Tuned Model

Same index, different model. Because `test_files` is built as `test_files_O[:50] + test_files_R[:50]`,
index 1 is an **organic** image — the first 50 entries are all class O.

Comparing both models on the identical image is the useful part: if they disagree, that single case
says more about what fine-tuning changed than either classification report does.

In [ ]:
# Task 10: Plot a test image using Fine-Tuned Model (index_to_plot = 1)
index_to_plot = 1
plot_image_with_title(
    image=test_imgs[index_to_plot].astype('uint8'),
    model_name='Fine-Tuned Model',
    actual_label=test_labels[index_to_plot],
    predicted_label=predictions_fine_tune_model[index_to_plot],
    )

📝 **A wrong prediction here isn't a failure of the exercise.** With 160 images per epoch, individual
errors are expected. The informative question is *which* images get missed — visually ambiguous ones
(a paper food container is arguably both), or ones where the model has latched onto background rather
than object.

## 📊 Summary

I built two waste classifiers on the same VGG16 backbone — one with the base entirely frozen, one with
`block5_conv3` onward unfrozen — and compared them on held-out test images.

### Pipeline

| Stage | Implementation | Purpose |
|---|---|---|
| 📥 **Data** | Waste Classification set, `O`/`R` folders | Labels from directory structure |
| 🔄 **Augmentation** | ±10% shift, horizontal flip | Mild, matched to real framing variation |
| 🏗️ **Base** | VGG16, `include_top=False`, 150×150 | Feature map 4×4×512 → flatten → 8192 |
| ➕ **Head** | Dense(512) → Drop(0.3) → Dense(512) → Drop(0.3) → Dense(1, sigmoid) | Binary decision |
| 🎯 **Model A** | Base frozen, Adam | Feature extraction baseline |
| 🔬 **Model B** | `block5_conv3`+ trainable, RMSprop | Fine-tuned comparison |
| 📊 **Evaluation** | `classification_report`, 100 test images | Precision / recall / F1 per class |
| 🧪 **Inspection** | Individual images with predictions | Qualitative check |

### Key equations

$$\sigma(z) = \frac{1}{1 + e^{-z}} \qquad \text{(sigmoid — soft decision)}$$

$$\mathcal{L} = -\big[y\log(\hat{y}) + (1-y)\log(1-\hat{y})\big] \qquad \text{(binary cross-entropy)}$$

$$\eta(t) = 10^{-4} \cdot e^{-0.1t} \qquad \text{(exponential LR decay)}$$

$$150 \to 75 \to 37 \to 18 \to 9 \to 4 \qquad \text{(VGG16's five pooling stages)}$$

### 🎓 What I take away

1. **Transfer learning has two distinct modes, and this project runs both.** Feature extraction is
   safe and cheap; fine-tuning trades that safety for adaptability. Training both is the only way to
   know which the problem actually needed.
2. **Freeze first, then unfreeze selectively.** Unfreezing from a named layer rather than an index
   survives architecture changes and reads as intent rather than arithmetic.
3. **A learning-rate scheduler outranks the optimizer's own setting.** Whatever `learning_rate=` says
   in `compile()`, the scheduler overwrites it every epoch. Two mechanisms, one variable.
4. **Sigmoid + 0.5 threshold is a soft decision followed by a hard slice.** The threshold is a tunable
   operating point, not a constant of nature.
5. **`ModelCheckpoint` and `EarlyStopping` do different jobs.** Without `restore_best_weights`, the
   checkpoint file is the only thing standing between you and evaluating a worse model than you trained.
6. **Inference preprocessing must mirror training preprocessing exactly.** Forgetting `/255` produces
   confident garbage and no error message.

### ⚠️ Problems in the source, and what they cost

| # | Problem | Consequence | Fix |
|---|---|---|---|
| 1 | `LearningRateScheduler` overrides `Adam(1e-5)` | Actual LR is 1e-4 — 10× the compiled value | Match `initial_lrate` to the optimizer, or drop the scheduler |
| 2 | `steps_per_epoch=5` (160 images/epoch) | Far from convergence; `EarlyStopping` may fire on noise | Remove the cap for a full pass |
| 3 | `LossHistory_` prints `exp_decay(len(losses))` | Printed LR is next epoch's, not the one just used | Print `exp_decay(epoch)` |
| 4 | `display()` never imported | Works in Jupyter only, `NameError` as a script | `from IPython.display import display`, or `print()` |
| 5 | `input_shape = basemodel.output_shape[1]` | Assigned, unused, shadows an earlier variable | Delete the line |
| 6 | Download cell not idempotent | Re-downloads the archive on every run | Guard with `os.path.exists` |
| 7 | `Flatten` over 4×4×512 | 4.2M params in one Dense layer | `GlobalAveragePooling2D` → 16× fewer |

## 🧪 Sandbox

Space to experiment, roughly in order of expected value:

**1. Resolve the learning-rate conflict.** Establish what LR is actually in effect, then make the code
say so:

```python
# Option A — let the scheduler own it, and align the compile call
def exp_decay(epoch):
    initial_lrate = 1e-5      # now matches Task 5's Adam(1e-5)
    return initial_lrate * np.exp(-0.1 * epoch)

# Option B — drop the scheduler and let the optimizer's LR stand
callbacks_list_ = [loss_history_] + keras_callbacks   # no lrate_
```

**2. Train on the full dataset.** Remove `steps_per_epoch=5` so each epoch is a complete pass, and
raise `epochs`. This is the biggest single lever on accuracy here:

```python
extract_feat_model = model.fit(train_generator,
                               epochs=30,
                               callbacks=callbacks_list_,
                               validation_data=val_generator)
```

**3. Swap `Flatten` for `GlobalAveragePooling2D`.** Cuts the first Dense layer from ~4.2M parameters to
~262k. Does accuracy drop, hold, or improve? If it holds, the spatial information was never being used:

```python
output = vgg.layers[-1].output
output = tf.keras.layers.GlobalAveragePooling2D()(output)
basemodel = Model(vgg.input, output)
```

**4. Sweep the unfreeze depth.** Try `block5_conv1`, `block4_conv1`, or the whole base. Where does
extra trainable capacity stop helping and start overfitting?

**5. Move the decision threshold.** Replace the hardcoded 0.5 and plot precision and recall against it.
Which operating point would a recycling plant actually want?

```python
for thresh in [0.3, 0.4, 0.5, 0.6, 0.7]:
    preds = ['O' if x < thresh else 'R' for x in raw_predictions]
    print(thresh, metrics.classification_report(test_labels, preds, output_dict=True)['R'])
```

**6. Make the download idempotent.** Wrap it in `if not os.path.exists('o-vs-r-split'):` so re-running
the notebook doesn't re-fetch the archive.

**7. Add a confusion matrix.** More informative than the report alone — it shows the *direction* of
errors, which is what determines cost on a sorting line:

```python
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay
cm = confusion_matrix(test_labels, predictions_fine_tune_model, labels=['O', 'R'])
ConfusionMatrixDisplay(cm, display_labels=['O', 'R']).plot()
```

**8. Look at every misclassified image.** Collect the failures and display them together. Patterns in
the errors are usually more instructive than the aggregate score.

**9. Evaluate on the whole test set.** Part 10 uses only the first 50 of each class. Use
`test_generator` (built in Task 2, with `shuffle=False`) to score everything.

**10. Try a lighter backbone.** `MobileNetV2` has roughly a tenth of VGG16's parameters and was
designed for edge deployment — the realistic choice if this ran on a camera above a conveyor rather
than on a workstation.

In [ ]:
# 🧪 Sandbox — experiment freely